# 🛡️ SAFIR — Adim Adim Calisma (Jupyter Walkthrough)

Bu defter, **Saha Analiz ve Farkindalik Icin Yapay Zeka Destekli Karar Sistemi (SAFIR)** pipeline'ini asama asama calistirir ve her adimin ciktisi gorulur:

1. **Adaptive Frame Sampler (CPU)** — videodan yalnizca 'kanit karelerini' suzer (GPU tasarrufu).
2. **Temsili Kareler** — her olayin oncesi/zirvesi/sonrasi (pre/peak/post).
3. **VLM (Gorsel Dil Modeli)** — kareleri Turkce betimler + yapilandirilmis olaylar uretir.
4. **Olay Tespiti** — VLM ciktisindan tipli olaylar (model-tabanli, keyword yedekli).
5. **Hibrit Bellek / RAG** — ilgili ISG mevzuati.
6. **LangGraph Ajani** — muhakeme -> risk skoru, ozet, aksiyon onerileri (JSON).
7. **Otomatik Eskalasyon** — risk'e gore aksiyon kademesi (Human-on-the-Loop).
8. **Nihai Rapor** — sartname-uyumlu JSON (uctan uca entegre kosu).


## 0) Kurulum ve Ayarlar
Once proje kokunu Python yoluna ekleriz ve konfigurasyonu yukleriz.

- `USE_MOCK`: `True` -> tamamen offline (Gemini/GPU gerekmez, sabit ornek cikti). `False` -> `config.yaml`'daki backend (Gemini). Gemini icin `GEMINI_API_KEY` tanimli olmali.
- `USE_FAKE_RAG`: `True` -> agir embedding modelini (bge-m3, ~2GB) indirmez; hafif sahte mevzuat kullanir (demo icin hizli). `False` -> gercek FAISS RAG.

In [ ]:
import sys, os
from pathlib import Path

# Proje kokunu bul (notebook notebooks/ altinda ya da kok dizinde olabilir)
_here = Path.cwd()
PROJECT_ROOT = _here if (_here / 'src').exists() else _here.parent
assert (PROJECT_ROOT / 'src').exists(), 'Proje koku bulunamadi (safir-ai/).'
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

# ---- AYARLAR ----
USE_MOCK = False       # True: offline | False: Gemini (config.yaml)
USE_FAKE_RAG = True    # True: bge-m3 indirmez (hizli) | False: gercek FAISS RAG

from src.utils.config_loader import load_config
config = load_config()
if USE_MOCK:
    config = config.model_copy(update={'app': config.app.model_copy(
        update={'use_mock_vlm': True, 'use_mock_llm': True})})
print('VLM backend :', config.vlm.active_model)
print('LLM backend :', config.llm.active_model)
print('MOCK        :', USE_MOCK, '| FAKE_RAG:', USE_FAKE_RAG)

### Video kaynagi
Kendi videon icin `VIDEO_PATH`'i `data/` altindaki dosyaya ayarla. Yoksa asagidaki hucre kucuk bir **sentetik** video uretir (hareketli bir dikdortgen) — pipeline'i gostermek icin yeterli.

In [ ]:
import cv2, numpy as np, tempfile

VIDEO_PATH = 'data/ornek.mp4'   # <-- kendi videon varsa buraya
if not Path(VIDEO_PATH).exists():
    _tmp = tempfile.mkdtemp()
    VIDEO_PATH = str(Path(_tmp) / 'synthetic.mp4')
    frames = [np.full((120, 160, 3), 30, np.uint8) for _ in range(60)]
    for i in range(20, 38):
        cv2.rectangle(frames[i], (20, 20), (140, 100), (210, 210, 210), -1)
    _w = cv2.VideoWriter(VIDEO_PATH, cv2.VideoWriter_fourcc(*'mp4v'), 25.0, (160, 120))
    for f in frames: _w.write(f)
    _w.release()
    print('Sentetik video uretildi:', VIDEO_PATH)
else:
    print('Video:', VIDEO_PATH)

## 1) Adaptive Frame Sampler (CPU)
Video tamamen CPU'da taranir; gurultu tabani dusulmus piksel degisimine gore yalnizca **kanit kareleri** secilir. Boylece VLM'e (pahali kismi) yalnizca onemli kareler gider — GPU tasarrufu buradan gelir.

In [ ]:
from src.sampler.adaptive_sampler import sampler_from_config

sampler = sampler_from_config(config.sampler)
evidence = sampler.process_video(VIDEO_PATH, sample_fps=config.sampler.sample_fps)
stats = sampler.last_run_stats
print(f'Kanit karesi sayisi : {len(evidence)}')
print(f'Taranan ham kare    : {stats.total_frames_scanned}')
print(f'Elenen kare         : {stats.eliminated_frame_count}')
print(f'GPU tasarruf orani  : %{stats.eliminated_ratio_pct}')
print(f'Sure                : {stats.elapsed_sec}s')

In [ ]:
# Kanit karelerini gorsel olarak goster
import base64
from IPython.display import Image as IPyImage, display

for ef in evidence:
    _, b64 = ef.base64_image.split(',', 1)
    print(f"[{ef.timestamp_str}] change_score={ef.change_score:.4f} fallback={ef.is_fallback}")
    display(IPyImage(data=base64.b64decode(b64), width=240))

## 2) Olay Kumeleme + Temsili Kareler (pre / peak / post)
Ardisik kanit kareleri **Olay Gruplari**na kumelenir. Her grup icin, zirve karenin oncesi ve sonrasindan da kare cikarilir; boylece VLM tek durgun kare yerine kisa bir **dizi** gorup olayin akisini (baslangic->gelisim->sonuc) muhakeme edebilir.

In [ ]:
from src.sampler.context.representative_frame_extractor import RepresentativeFrameExtractor

clusters = sampler.cluster_events(evidence)
extractor = RepresentativeFrameExtractor(
    config.sampler.pre_peak_offset_sec, config.sampler.post_peak_offset_sec)
for c in clusters:
    c.representative_frames = extractor.extract(VIDEO_PATH, c.peak_frame)

print(f'{len(clusters)} olay grubu')
for c in clusters:
    roles = [(rf.label, rf.timestamp_str) for rf in c.representative_frames]
    print(f'  Olay #{c.event_id}: {roles}')

## 3) VLM — Gorsel Anlama
Secilen kareler VLM'e gonderilir (Gemini veya mock). VLM iki sey uretir:
- **Insan-okur** Turkce sahne gozlemi,
- **Makine-okur** yapilandirilmis olaylar (`EVENTS_JSON`: tip/zaman/guven).

In [ ]:
from src.vlm.factory import get_vlm_client

vlm = get_vlm_client(config.vlm, use_mock=config.app.use_mock_vlm)
vlm_response = vlm.describe_events(clusters, prompt='Sahnede riskli bir durum var mi degerlendir.')

print('Model:', vlm_response.model_name)
print('\n--- VLM Gozlemi ---\n')
print(vlm_response.description)
print('\n--- Yapilandirilmis olaylar (EVENTS_JSON) ---')
print(vlm_response.structured_events)

## 4) Olay Tespiti (EventEngine)
VLM ciktisi tipli `DetectedEvent`lere cevrilir. **Once** VLM'in dogrudan urettigi yapilandirilmis olaylar kullanilir (model-tabanli); yoksa anahtar-kelime + olumsuzlama tespiti **yedek** olarak devreye girer.

In [ ]:
from src.event_analysis.event_engine import EventEngine
from src.event_analysis.schemas import EventEngineInput

engine_input = EventEngineInput.from_vlm_response(vlm_response, timestamp=clusters[-1].end_time)
detected = EventEngine().detect(engine_input)
for d in detected:
    print(f'  {d.event_type:<24} guven={d.confidence:.2f}  keywords={d.matched_keywords}')

## 5) Hibrit Bellek / RAG — Ilgili ISG Mevzuati
Gozlemle ilgili mevzuat maddeleri anlamsal arama (Embedding + FAISS) ile getirilir. `USE_FAKE_RAG=True` iken hafif sahte mevzuat kullanilir (agir model indirilmez).

In [ ]:
if USE_FAKE_RAG:
    from dataclasses import dataclass
    @dataclass
    class _Doc:
        text: str
        score: float = 1.0
    class _FakeRAG:
        def seed_default_regulations(self): pass
        def query(self, q, top_k=None):
            return [
                _Doc('ISG Yonetmeligi Madde 24: KKD (baret/yelek) zorunludur.'),
                _Doc('Operasyonel Kural OK-07: Forklift trafiginde yaya gecitleri acik tutulmalidir.'),
            ][:(top_k or 2)]
    rag_service = _FakeRAG()
else:
    from src.memory.embedding_rag_service import EmbeddingRAGService
    rag_service = EmbeddingRAGService(config.memory.embedding, config.memory.faiss)
    rag_service.seed_default_regulations()

for doc in rag_service.query('KKD eksikligi ve forklift yaya yakinligi', top_k=2):
    print(' -', doc.text)

## 6) LangGraph Ajani — Muhakeme ve Karar
Ajan; gozlemi, mevzuati ve olay sinyallerini alir, gerekirse araclarini (sql / retriever / timeline / verification) cagirir ve **sartname-uyumlu JSON** bir karar uretir: risk skoru, seviye, ozet ve aksiyon onerileri.

In [ ]:
from src.agent.langgraph_agent import SafirAgent

agent = SafirAgent(
    llm_config=config.llm, agent_config=config.agent,
    event_store=None, rag_service=rag_service, use_mock_llm=config.app.use_mock_llm)

context = (
    f'## Guncel Gozlem\n{vlm_response.description}\n\n'
    '## Kullanici Istemi\nSahnede riskli bir durum var mi degerlendir.'
)
decision = agent.run(context)

print('Risk skoru :', decision.risk_score, f'({decision.risk_level})')
print('Ozet       :', decision.summary)
print('Aksiyonlar :', decision.actions)

## 7) Otomatik Eskalasyon (Human-on-the-Loop)
Bloke edici bir operator onayi YOKTUR: sistem risk skoruna gore aksiyon kademesini kendisi secer. Yuksek/kritik riskte saha alarmi **otomatik** tetiklenir; operator sonradan denetler.

In [ ]:
from src.decision.escalation import EscalationPolicy

policy = EscalationPolicy(config.escalation)
esc = policy.evaluate(
    risk_score=decision.risk_score, risk_level=decision.risk_level,
    recommended_action=decision.recommended_action, summary=decision.summary)

print('Kademe          :', esc.tier.value)
print('Otomatik alarm  :', esc.auto_dispatched)
print('alert_id        :', esc.alert_id)
print('Gerekce         :', esc.reason)

## 8) Uctan Uca Entegre Kosu — Nihai Rapor
Yukaridaki tum asamalar `SafirPipeline.run()` icinde birlesir. Tek cagriyla nihai **sartname-uyumlu JSON** raporu uretilir (ozet, olaylar, risk, aksiyonlar).

In [ ]:
import json
import src.main as safir_main

if USE_FAKE_RAG:
    safir_main.EmbeddingRAGService = lambda *a, **k: rag_service  # agir modeli indirme

pipeline = safir_main.SafirPipeline(config)
report = pipeline.run(VIDEO_PATH, 'Sahnede riskli bir durum var mi degerlendir.')

print('risk         :', report.risk_score, f'({report.risk_level})')
print('eskalasyon   :', report.escalation_tier, '| otomatik:', report.auto_dispatched)
print('tespit tipler:', report.detected_event_types)
print('\n--- SARTNAME UYUMLU JSON ---')
print(json.dumps(report.to_sartname_json(), ensure_ascii=False, indent=2))

---
### Ozet
Bu defter, SAFIR'in **CPU suzgec -> VLM -> olay tespiti -> RAG -> ajan -> otomatik eskalasyon -> rapor** akisini adim adim gosterdi. Gercek Gemini ciktisi icin `USE_MOCK=False` (ve `GEMINI_API_KEY`), tam offline demo icin `USE_MOCK=True` yeterlidir. Operator paneli (Streamlit) ayni backend'i kullanir: `streamlit run src/ui/dashboard.py`.